In [ ]:
# Boot cell: makes this notebook behave exactly like `python labs/08_04_05_rag_evals.py` run from the repo root.
# A Jupyter kernel has no __file__, starts in the labs folder and carries its own sys.argv; the script expects none of that.
import os, sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").is_file() and (p / "labs").is_dir())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
__file__ = str(_root / "labs" / "08_04_05_rag_evals.py")
sys.argv = [__file__]
print("repo root:", _root)


In [ ]:
"""Lab 25.5: RAG evals on the DataFlow question file.

What we build: you type the small function that scores one answer on refusal: did it refuse with the house
sentence, was that right for this row, is it a fluent miss (an answer to a question the knowledge base does
not cover), or an over-refusal (a refusal on a row that has an answer). You check it on four answers with
the real judge, then run the twelve DataFlow questions through two desks, one that retrieves and answers
every time and Section 24's graph that grades and can refuse, and read the table.

What you learn: 25.3's idea, measured. Refusing on empty is worth something only when you can count it, so
the fluent miss becomes a number per desk; the grader has a price you can read in the latency, token and
call columns. And the limits you see: the judge is the desk's own model, a passage judge cannot score a
lookup, and a refusal in the model's own words does not count as the house refusal.

What will be true on your screen: four refusal_scores lines (one an own-words refusal flagged as a fluent
miss), warmup_s, the summary table with tokens_per_q and calls_per_q, the per-row lines, the naive desk's
answers on the three refuse rows, over_refusals per desk, recall_by_definition 5 of 12, and
judge_is_desk_model True. Then python -m pytest tests/test_lab_08_04_05.py -q passes.
"""

from pathlib import Path
import sys
import time

root = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()
sys.path.insert(0, str(root))

import config
from dataflow.graphs.rag_graph import REFUSE_TEMPLATE
from eval.runners.rag_metrics import QUESTIONS, _refused, faithfulness_judge, load_questions, render, run_metrics


In [ ]:
# TYPE THIS: refusal_scores. The signature and docstring are given; you type the body.
# Its finished copy is refusal_scores in eval/runners/rag_metrics.py, which score_row calls.
def refusal_scores(kind: str, answer: str, passages: list, *, judge, model=None) -> dict:
    """Score one answer on refusal. refused is True only when the answer carries the house refusal sentence. A refuse row that refused: refused_correctly 1, faithfulness 1, no judge call. A refuse row that answered: refused_correctly 0, faithfulness from the judge, fluent_miss True when the answer is not empty. Any other row: a refusal scores faithfulness 1 and over_refusal True; an answer goes to the judge."""
    refused = _refused(answer)
    return {
        "refused": refused,
        "refused_correctly": 1 if kind == "refuse" and refused else 0,
        "faithfulness": 1 if refused else judge(answer, passages, model=model),
        "fluent_miss": kind == "refuse" and not refused and bool(answer.strip()),
        "over_refusal": kind != "refuse" and refused,
    }


# BREAK: drop the row-kind guard from fluent_miss, so it reads
#         "fluent_miss": not refused and bool(answer.strip()),
# run the lab and read the policy row's line, then python -m pytest tests/test_lab_08_04_05.py -q.
# git checkout -- labs/08_04_05_rag_evals.py puts it back.


In [ ]:
chat = config.get_chat_model()
print("model", type(chat).__name__, getattr(chat, "model", None) or getattr(chat, "model_name", None))
judged_with = []


def judge(answer, passages, model=None):
    """The faithfulness judge from rag_metrics, noting which model it was handed."""
    judged_with.append(model)
    return faithfulness_judge(answer, passages, model=model)


house = REFUSE_TEMPLATE.format(q="Do you offer a lifetime warranty on every desk lamp?")
for kind, answer in (
    ("refuse", house),
    ("refuse", "Yes, every desk lamp has a lifetime warranty."),
    ("refuse", "I don't have information about a lifetime warranty."),
    ("policy", house),
):
    print("refusal_scores", kind, repr(answer[:60]), refusal_scores(kind, answer, [], judge=judge, model=chat))


In [ ]:
# One warm-up call so the first row does not carry the model's first load.
started = time.perf_counter()
chat.invoke("Reply with the word ready.")
print("warmup_s", round(time.perf_counter() - started, 1))
print("questions", QUESTIONS.relative_to(config.ROOT).as_posix())
report = run_metrics(model=chat, judge=judge)
print(render(report))


In [ ]:
rows = {row["id"]: row for row in load_questions()}
for scored in report["naive"]:
    if scored["kind"] == "refuse":
        print("naive_answer", scored["id"], repr(str(scored["answer"])[:160]))
over = {name: sum(1 for row in report[name] if row["kind"] != "refuse" and _refused(row["answer"])) for name in ("naive", "agentic")}
print("over_refusals", "naive", over["naive"], "agentic", over["agentic"])
print("recall_by_definition", sum(1 for row in rows.values() if not row.get("expected_source")), "of", len(rows))
print("judge_model", type(chat).__name__, getattr(chat, "model", None) or getattr(chat, "model_name", None))
print("judge_is_desk_model", bool(judged_with) and all(model is chat for model in judged_with))
